# Chapter 4 Companion Notebook: Meridian Fabrication Inc. Financial Statement Analysis

This notebook reproduces the Meridian Fabrication Inc. running example from Chapter 4 of *AI in Finance* (`chapter4.tex`), recomputing every ratio and figure quoted in the text directly from the statements, plus the Solstice Robotics comparison, common-size income and balance-sheet statements, EBITDA and ROIC-vs-WACC economic profit, three- and five-factor DuPont decomposition, FCFF vs. FCFE, the Altman Z'-score for both companies, and a complete percent-of-sales forecast through projected Year 3 free cash flow.

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. The three statements

In [1]:
import pandas as pd

balance_sheet = pd.DataFrame({
    'Year 1': {
        'Cash': 12.0, 'Accounts receivable': 18.0, 'Inventory': 22.0,
        'PP&E (net)': 60.0,
        'Accounts payable': 14.0, 'Short-term debt': 6.0, 'Long-term debt': 40.0,
        'Equity': 52.0,
    },
    'Year 2': {
        'Cash': 9.0, 'Accounts receivable': 24.0, 'Inventory': 30.0,
        'PP&E (net)': 66.0,
        'Accounts payable': 17.0, 'Short-term debt': 8.0, 'Long-term debt': 50.0,
        'Equity': 54.0,
    },
}).T

balance_sheet['Total current assets'] = (
    balance_sheet['Cash'] + balance_sheet['Accounts receivable'] + balance_sheet['Inventory']
)
balance_sheet['Total assets'] = balance_sheet['Total current assets'] + balance_sheet['PP&E (net)']
balance_sheet['Total current liabilities'] = (
    balance_sheet['Accounts payable'] + balance_sheet['Short-term debt']
)
balance_sheet['Total liabilities'] = balance_sheet['Total current liabilities'] + balance_sheet['Long-term debt']

# sanity check: Assets = Liabilities + Equity
check = balance_sheet['Total assets'] - (balance_sheet['Total liabilities'] + balance_sheet['Equity'])
assert (check.abs() < 1e-9).all(), "Balance sheet does not balance!"
balance_sheet.round(1)

,Cash,Accounts receivable,Inventory,PP&E (net),Accounts payable,Short-term debt,Long-term debt,Equity,Total current assets,Total assets,Total current liabilities,Total liabilities
Year 1,12.0,18.0,22.0,60.0,14.0,6.0,40.0,52.0,52.0,112.0,20.0,60.0
Year 2,9.0,24.0,30.0,66.0,17.0,8.0,50.0,54.0,63.0,129.0,25.0,75.0


In [2]:
income_statement = {
    'Revenue': 150.0, 'COGS': 96.0, 'SG&A': 30.0, 'D&A': 8.0,
    'Interest expense': 4.0, 'Tax rate': 0.25,
}
gross_profit = income_statement['Revenue'] - income_statement['COGS']
ebit = gross_profit - income_statement['SG&A'] - income_statement['D&A']
pretax_income = ebit - income_statement['Interest expense']
taxes = pretax_income * income_statement['Tax rate']
net_income = pretax_income - taxes

print(f"Gross profit:  ${gross_profit:.1f}m")
print(f"EBIT:          ${ebit:.1f}m")
print(f"Pretax income: ${pretax_income:.1f}m")
print(f"Taxes:         ${taxes:.1f}m")
print(f"Net income:    ${net_income:.1f}m")

Gross profit:  $54.0m
EBIT:          $16.0m
Pretax income: $12.0m
Taxes:         $3.0m
Net income:    $9.0m


### 1b. EBITDA and ROIC vs. WACC (Section 4.9, Section 4.10)

EBITDA adds depreciation and amortization back to EBIT. Economic profit compares return on invested capital (ROIC) to the weighted average cost of capital (WACC).

In [3]:
ebitda = ebit + income_statement['D&A']
ebitda_margin = ebitda / income_statement['Revenue']
print(f"EBITDA: ${ebitda:.1f}m, EBITDA margin: {ebitda_margin:.1%}")

y2_bs = balance_sheet.loc['Year 2']
total_debt = y2_bs['Short-term debt'] + y2_bs['Long-term debt']
invested_capital = total_debt + y2_bs['Equity']
nopat = ebit * (1 - income_statement['Tax rate'])
roic = nopat / invested_capital

for wacc in [0.08, 0.12]:
    economic_profit = invested_capital * (roic - wacc)
    print(f"WACC={wacc:.0%}: ROIC={roic:.2%}, invested capital=${invested_capital:.1f}m, "
          f"economic profit=${economic_profit:.2f}m")

EBITDA: $24.0m, EBITDA margin: 16.0%
WACC=8%: ROIC=10.71%, invested capital=$112.0m, economic profit=$3.04m
WACC=12%: ROIC=10.71%, invested capital=$112.0m, economic profit=$-1.44m


## 2. Free cash flow

Year 2 cash flow from operations is \$6.0m (net income + D&A, less working-capital increases, plus payables increase); capital expenditures are \$14.0m.

In [4]:
cfo = 6.0
capex = 14.0
free_cash_flow = cfo - capex
print(f"Free cash flow: ${free_cash_flow:.1f}m")
print(f"Net income was ${net_income:.1f}m -- FCF is negative despite positive net income.")

Free cash flow: $-8.0m
Net income was $9.0m -- FCF is negative despite positive net income.


### 2b. FCFF vs. FCFE (Section 4.17)

Free cash flow to the firm (all capital providers) vs. free cash flow to equity (equity holders only, after net borrowing).

In [5]:
net_borrowing = 12.0
fcff = cfo + income_statement['Interest expense'] * (1 - income_statement['Tax rate']) - capex
fcfe = cfo - capex + net_borrowing
print(f"FCFF: ${fcff:.1f}m")
print(f"FCFE: ${fcfe:.1f}m")

# Exercise: net borrowing of $2.0m instead of $12.0m
net_borrowing_ex = 2.0
fcfe_ex = cfo - capex + net_borrowing_ex
print(f"\nExercise (net borrowing=$2.0m) -- FCFF: ${fcff:.1f}m (unchanged), FCFE: ${fcfe_ex:.1f}m")

FCFF: $-5.0m
FCFE: $4.0m

Exercise (net borrowing=$2.0m) -- FCFF: $-5.0m (unchanged), FCFE: $-6.0m


## 3. Ratio analysis (Year 2)

Reproduces the ratio summary table from Section 4.11 ("Common Financial Ratios and What They Reveal").

In [6]:
y2 = balance_sheet.loc['Year 2']

current_ratio = y2['Total current assets'] / y2['Total current liabilities']
quick_ratio = (y2['Total current assets'] - 30.0) / y2['Total current liabilities']
gross_margin = gross_profit / income_statement['Revenue']
operating_margin = ebit / income_statement['Revenue']
net_margin = net_income / income_statement['Revenue']
total_debt = y2['Short-term debt'] + y2['Long-term debt']
debt_to_equity = total_debt / y2['Equity']
interest_coverage = ebit / income_statement['Interest expense']
asset_turnover = income_statement['Revenue'] / y2['Total assets']
roe = net_income / y2['Equity']

ratios = pd.Series({
    'Current ratio': current_ratio,
    'Quick ratio': quick_ratio,
    'Gross margin': gross_margin,
    'Operating margin': operating_margin,
    'Net margin': net_margin,
    'Debt-to-equity': debt_to_equity,
    'Interest coverage': interest_coverage,
    'Asset turnover': asset_turnover,
    'Return on equity': roe,
})
ratios.round(4)

Current ratio        2.5200
Quick ratio          1.3200
Gross margin         0.3600
Operating margin     0.1067
Net margin           0.0600
Debt-to-equity       1.0741
Interest coverage    4.0000
Asset turnover       1.1628
Return on equity     0.1667
dtype: float64

## 4. DuPont decomposition of ROE

$\text{ROE} = \text{Net margin} \times \text{Asset turnover} \times \text{Equity multiplier}$

In [7]:
equity_multiplier = y2['Total assets'] / y2['Equity']
dupont_roe = net_margin * asset_turnover * equity_multiplier

print(f"Net margin:        {net_margin:.4f}")
print(f"Asset turnover:    {asset_turnover:.4f}")
print(f"Equity multiplier: {equity_multiplier:.4f}")
print(f"DuPont ROE:        {dupont_roe:.4%}")
print(f"Direct ROE:        {roe:.4%}")

Net margin:        0.0600
Asset turnover:    1.1628
Equity multiplier: 2.3889
DuPont ROE:        16.6667%
Direct ROE:        16.6667%


### 4b. Five-factor extended DuPont (Section 4.11)

Split net margin into tax burden x interest burden x operating margin.

In [8]:
tax_burden = net_income / pretax_income
interest_burden = pretax_income / ebit
operating_margin = ebit / income_statement['Revenue']

roe_5factor = tax_burden * interest_burden * operating_margin * asset_turnover * equity_multiplier
print(f"Tax burden: {tax_burden:.4f}")
print(f"Interest burden: {interest_burden:.4f}")
print(f"Operating margin: {operating_margin:.4f}")
print(f"5-factor ROE: {roe_5factor:.4%} (direct ROE: {roe:.4%})")

Tax burden: 0.7500
Interest burden: 0.7500
Operating margin: 0.1067
5-factor ROE: 16.6667% (direct ROE: 16.6667%)


## 5. Cash conversion cycle

In [9]:
dio = y2['Inventory'] / income_statement['COGS'] * 365
dso = y2['Accounts receivable'] / income_statement['Revenue'] * 365
dpo = y2['Accounts payable'] / income_statement['COGS'] * 365
ccc = dio + dso - dpo

print(f"DIO: {dio:.0f} days")
print(f"DSO: {dso:.0f} days")
print(f"DPO: {dpo:.0f} days")
print(f"Cash conversion cycle: {ccc:.0f} days")

DIO: 114 days
DSO: 58 days
DPO: 65 days
Cash conversion cycle: 108 days


## 6. A second company for comparison: Solstice Robotics Inc.

Solstice is a faster-growing, less capital-intensive firm. Compute its Year 2 ratios and DuPont decomposition, and compare to Meridian.

In [10]:
solstice = {
    'Cash': 40.0, 'Accounts receivable': 20.0, 'Inventory': 10.0,
    'PP&E (net)': 30.0,
    'Accounts payable': 15.0, 'Long-term debt': 10.0, 'Equity': 75.0,
    'Revenue': 120.0, 'COGS': 48.0, 'SG&A': 40.0, 'D&A': 5.0, 'Interest expense': 1.0,
    'Net income': 19.5,
}
solstice['Total current assets'] = solstice['Cash'] + solstice['Accounts receivable'] + solstice['Inventory']
solstice['Total assets'] = solstice['Total current assets'] + solstice['PP&E (net)']
solstice['Total liabilities'] = solstice['Accounts payable'] + solstice['Long-term debt']

s_current_ratio = solstice['Total current assets'] / solstice['Accounts payable']
s_quick_ratio = (solstice['Total current assets'] - solstice['Inventory']) / solstice['Accounts payable']
s_gross_margin = (solstice['Revenue'] - solstice['COGS']) / solstice['Revenue']
s_ebit = solstice['Revenue'] - solstice['COGS'] - solstice['SG&A'] - solstice['D&A']
s_operating_margin = s_ebit / solstice['Revenue']
s_net_margin = solstice['Net income'] / solstice['Revenue']
s_debt_to_equity = solstice['Long-term debt'] / solstice['Equity']
s_interest_coverage = s_ebit / solstice['Interest expense']
s_asset_turnover = solstice['Revenue'] / solstice['Total assets']
s_equity_multiplier = solstice['Total assets'] / solstice['Equity']
s_roe = solstice['Net income'] / solstice['Equity']
s_dupont_roe = s_net_margin * s_asset_turnover * s_equity_multiplier

comparison = pd.DataFrame({
    'Meridian': [current_ratio, quick_ratio, gross_margin, operating_margin, net_margin,
                 debt_to_equity, interest_coverage, roe],
    'Solstice': [s_current_ratio, s_quick_ratio, s_gross_margin, s_operating_margin, s_net_margin,
                 s_debt_to_equity, s_interest_coverage, s_roe],
}, index=['Current ratio', 'Quick ratio', 'Gross margin', 'Operating margin', 'Net margin',
          'Debt-to-equity', 'Interest coverage', 'Return on equity'])
print(comparison.round(4))

print(f"\nSolstice DuPont ROE: {s_net_margin:.4f} x {s_asset_turnover:.4f} x {s_equity_multiplier:.4f} = {s_dupont_roe:.4%}")
print(f"Solstice direct ROE: {s_roe:.4%}")

                   Meridian  Solstice
Current ratio        2.5200    4.6667
Quick ratio          1.3200    4.0000
Gross margin         0.3600    0.6000
Operating margin     0.1067    0.2250
Net margin           0.0600    0.1625
Debt-to-equity       1.0741    0.1333
Interest coverage    4.0000   27.0000
Return on equity     0.1667    0.2600

Solstice DuPont ROE: 0.1625 x 1.2000 x 1.3333 = 26.0000%
Solstice direct ROE: 26.0000%


In [11]:
# Exercise: Solstice's five-factor DuPont
s_pretax = 20.5
s_tax_burden = solstice['Net income'] / s_pretax
s_interest_burden = s_pretax / s_ebit
s_roe_5factor = s_tax_burden * s_interest_burden * s_operating_margin * s_asset_turnover * s_equity_multiplier
print(f"Exercise (Solstice 5-factor DuPont) -- tax burden: {s_tax_burden:.4f}, interest burden: {s_interest_burden:.4f}")
print(f"Solstice 5-factor ROE: {s_roe_5factor:.4%} (direct ROE: {s_roe:.4%})")

Exercise (Solstice 5-factor DuPont) -- tax burden: 0.9512, interest burden: 0.7593
Solstice 5-factor ROE: 26.0000% (direct ROE: 26.0000%)


## 7. Common-size income statements

Express every income statement line as a percent of revenue for both firms.

In [12]:
common_size = pd.DataFrame({
    'Meridian': {
        'COGS': income_statement['COGS'] / income_statement['Revenue'],
        'Gross profit': gross_profit / income_statement['Revenue'],
        'SG&A': income_statement['SG&A'] / income_statement['Revenue'],
        'D&A': income_statement['D&A'] / income_statement['Revenue'],
        'EBIT': ebit / income_statement['Revenue'],
        'Interest expense': income_statement['Interest expense'] / income_statement['Revenue'],
        'Net income': net_income / income_statement['Revenue'],
    },
    'Solstice': {
        'COGS': solstice['COGS'] / solstice['Revenue'],
        'Gross profit': (solstice['Revenue'] - solstice['COGS']) / solstice['Revenue'],
        'SG&A': solstice['SG&A'] / solstice['Revenue'],
        'D&A': solstice['D&A'] / solstice['Revenue'],
        'EBIT': s_ebit / solstice['Revenue'],
        'Interest expense': solstice['Interest expense'] / solstice['Revenue'],
        'Net income': solstice['Net income'] / solstice['Revenue'],
    },
})
(common_size * 100).round(1)

,Meridian,Solstice
COGS,64.0,40.0
Gross profit,36.0,60.0
SG&A,20.0,33.3
D&A,5.3,4.2
EBIT,10.7,22.5
Interest expense,2.7,0.8
Net income,6.0,16.2


### 7b. Common-size balance sheet (Section 4.13)

Express Meridian's Year 2 balance sheet lines as a percent of total assets.

In [13]:
cs_balance_sheet = (y2[['Cash', 'Accounts receivable', 'Inventory', 'Total current assets', 'PP&E (net)',
                        'Accounts payable', 'Short-term debt', 'Total current liabilities',
                        'Long-term debt', 'Total liabilities', 'Equity']] / y2['Total assets'] * 100)
cs_balance_sheet.round(2)

Cash                          6.98
Accounts receivable          18.60
Inventory                    23.26
Total current assets         48.84
PP&E (net)                   51.16
Accounts payable             13.18
Short-term debt               6.20
Total current liabilities    19.38
Long-term debt               38.76
Total liabilities            58.14
Equity                       41.86
Name: Year 2, dtype: float64

## 8. A simple distress score: Altman's Z' (private-firm variant)

Assumes all of Meridian's Year 2 equity (\$54.0m) is retained earnings.

In [14]:
working_capital = y2['Total current assets'] - y2['Total current liabilities']
retained_earnings = y2['Equity']  # assumed, for illustration

X1 = working_capital / y2['Total assets']
X2 = retained_earnings / y2['Total assets']
X3 = ebit / y2['Total assets']
X4 = y2['Equity'] / y2['Total liabilities']
X5 = income_statement['Revenue'] / y2['Total assets']

Z_prime = 0.717 * X1 + 0.847 * X2 + 3.107 * X3 + 0.420 * X4 + 0.998 * X5

print(f"X1 (WC/TA): {X1:.3f}, X2 (RE/TA): {X2:.3f}, X3 (EBIT/TA): {X3:.3f}, "
      f"X4 (Equity/TL): {X4:.3f}, X5 (Sales/TA): {X5:.3f}")
print(f"Z' = {Z_prime:.2f}")
zone = "safe" if Z_prime > 2.9 else ("grey" if Z_prime > 1.23 else "distress")
print(f"Zone: {zone}")

X1 (WC/TA): 0.295, X2 (RE/TA): 0.419, X3 (EBIT/TA): 0.124, X4 (Equity/TL): 0.720, X5 (Sales/TA): 1.163
Z' = 2.41
Zone: grey


### 8b. Solstice's Altman Z' for comparison

In [15]:
s_working_capital = solstice['Total current assets'] - solstice['Accounts payable']
s_retained_earnings = solstice['Equity']  # assumed

sX1 = s_working_capital / solstice['Total assets']
sX2 = s_retained_earnings / solstice['Total assets']
sX3 = s_ebit / solstice['Total assets']
sX4 = solstice['Equity'] / solstice['Total liabilities']
sX5 = solstice['Revenue'] / solstice['Total assets']

s_Z_prime = 0.717 * sX1 + 0.847 * sX2 + 3.107 * sX3 + 0.420 * sX4 + 0.998 * sX5

print(f"X1: {sX1:.3f}, X2: {sX2:.3f}, X3: {sX3:.3f}, X4: {sX4:.3f}, X5: {sX5:.3f}")
print(f"Solstice Z' = {s_Z_prime:.2f}")
print(f"Meridian Z' = {Z_prime:.2f}")

X1: 0.550, X2: 0.750, X3: 0.270, X4: 3.000, X5: 1.200
Solstice Z' = 4.33
Meridian Z' = 2.41


## 9. Forecasting from statements: a percent-of-sales model

Project Year 3 under 8% revenue growth, holding percent-of-revenue (or percent-of-COGS) ratios constant, with interest expense held fixed in dollar terms.

In [16]:
growth = 0.08
rev3 = income_statement['Revenue'] * (1 + growth)

cogs_pct = income_statement['COGS'] / income_statement['Revenue']
sga_pct = income_statement['SG&A'] / income_statement['Revenue']
da_pct = income_statement['D&A'] / income_statement['Revenue']
ar_pct = y2['Accounts receivable'] / income_statement['Revenue']
inv_pct = y2['Inventory'] / income_statement['Revenue']
ap_pct_cogs = y2['Accounts payable'] / income_statement['COGS']

cogs3 = rev3 * cogs_pct
gross_profit3 = rev3 - cogs3
sga3 = rev3 * sga_pct
da3 = rev3 * da_pct
ebit3 = gross_profit3 - sga3 - da3
interest3 = income_statement['Interest expense']  # held constant
pretax3 = ebit3 - interest3
taxes3 = pretax3 * income_statement['Tax rate']
net_income3 = pretax3 - taxes3

ar3 = rev3 * ar_pct
inv3 = rev3 * inv_pct
ap3 = cogs3 * ap_pct_cogs

forecast = pd.Series({
    'Revenue': rev3, 'COGS': cogs3, 'Gross profit': gross_profit3, 'SG&A': sga3,
    'D&A': da3, 'EBIT': ebit3, 'Interest expense': interest3, 'Pretax income': pretax3,
    'Taxes': taxes3, 'Net income': net_income3,
    'Accounts receivable': ar3, 'Inventory': inv3, 'Accounts payable': ap3,
})
print(forecast.round(2))

ni_growth = (net_income3 - net_income) / net_income
print(f"\nNet income growth: {ni_growth:.1%} (vs. {growth:.0%} revenue growth)")

Revenue                162.00
COGS                   103.68
Gross profit            58.32
SG&A                    32.40
D&A                      8.64
EBIT                    17.28
Interest expense         4.00
Pretax income           13.28
Taxes                    3.32
Net income               9.96
Accounts receivable     25.92
Inventory               32.40
Accounts payable        18.36
dtype: float64

Net income growth: 10.7% (vs. 8% revenue growth)


### 9b. Projected Year 3 free cash flow (Section 4.16)

Complete the forecast by projecting Year 3 cash flow from operations and capital expenditures (held at their Year 2 percent-of-revenue).

In [17]:
d_ar3 = ar3 - y2['Accounts receivable']
d_inv3 = inv3 - y2['Inventory']
d_ap3 = ap3 - y2['Accounts payable']
cfo3 = net_income3 + da3 - d_ar3 - d_inv3 + d_ap3

capex_pct = 14.0 / income_statement['Revenue']
capex3 = capex_pct * rev3
fcf3 = cfo3 - capex3

print(f"Year 3 CFO: ${cfo3:.2f}m")
print(f"Year 3 capex: ${capex3:.2f}m")
print(f"Year 3 FCF: ${fcf3:.2f}m (Year 2 FCF was ${free_cash_flow:.1f}m)")

Year 3 CFO: $15.64m
Year 3 capex: $15.12m
Year 3 FCF: $0.52m (Year 2 FCF was $-8.0m)


## Exercises (match Chapter 4, Suggested Exercises)

Use the `balance_sheet` and ratio-computation cells above as templates to:

1. Compute the current ratio, quick ratio, and debt-to-equity ratio for **Year 1** and compare them to the Year 2 values above.
2. Recompute free cash flow assuming capital expenditures of \$6.0m instead of \$14.0m.
3. Decompose Year 1 ROE with the DuPont identity, assuming Year 1 net income of \$7.5m and Year 1 revenue of \$135.0m.

In [18]:
# Exercise 1: Year 1 ratios
y1 = balance_sheet.loc['Year 1']
cr_y1 = y1['Total current assets'] / y1['Total current liabilities']
qr_y1 = (y1['Total current assets'] - 22.0) / y1['Total current liabilities']
de_y1 = (y1['Short-term debt'] + y1['Long-term debt']) / y1['Equity']
print(f"Year 1 -- current ratio: {cr_y1:.2f}, quick ratio: {qr_y1:.2f}, debt-to-equity: {de_y1:.2f}")

# Exercise 2: FCF with lower capex
fcf_alt = cfo - 6.0
print(f"Exercise 2 -- FCF with $6.0m capex: ${fcf_alt:.1f}m")

# Exercise 3: Year 1 DuPont
ni_y1, rev_y1 = 7.5, 135.0
nm_y1 = ni_y1 / rev_y1
at_y1 = rev_y1 / y1['Total assets']
em_y1 = y1['Total assets'] / y1['Equity']
roe_y1 = nm_y1 * at_y1 * em_y1
print(f"Exercise 3 -- Year 1 ROE via DuPont: {roe_y1:.4%} "
      f"(margin={nm_y1:.4f}, turnover={at_y1:.4f}, multiplier={em_y1:.4f})")

Year 1 -- current ratio: 2.60, quick ratio: 1.50, debt-to-equity: 0.88
Exercise 2 -- FCF with $6.0m capex: $0.0m
Exercise 3 -- Year 1 ROE via DuPont: 14.4231% (margin=0.0556, turnover=1.2054, multiplier=2.1538)
